# Transfer Learning: usar modelos preentrenados

Transfer Learning es una de las técnicas más poderosas en deep learning. Permite aprovechar modelos entrenados en millones de imágenes (ImageNet) y adaptarlos a nuestro problema específico con pocos datos y poco tiempo de entrenamiento.

**Contenido:**
1. Qué es Transfer Learning y por qué funciona
2. Estrategia 1: Feature Extraction (congelar todo, entrenar solo la cabeza)
3. Estrategia 2: Fine-Tuning (descongelar capas progresivamente)
4. Comparación de resultados
5. Tips prácticos y modelos alternativos

**Requisitos:**
```bash
pip install torch torchvision matplotlib
```

## Qué es Transfer Learning y por qué funciona

**Transfer Learning** consiste en tomar un modelo entrenado en una tarea (por ejemplo, clasificar 1000 categorías de ImageNet) y reutilizar sus conocimientos para una tarea diferente (por ejemplo, detectar defectos en piezas industriales).

**Por qué funciona:**
- Las primeras capas de una CNN aprenden features universales (bordes, texturas, patrones)
- Estas features son útiles para casi cualquier tarea de visión
- Solo las últimas capas son específicas a la tarea original

```
Capas iniciales:  Bordes → Texturas → Partes de objetos → Objetos completos  :Capas finales
   (genérico)                                                                  (específico)
```

**Cuándo usarlo:**
- Tenemos pocos datos (< 10,000 imágenes)
- Nuestro dominio es similar a ImageNet (fotos naturales)
- Queremos resultados rápidos sin entrenar días en GPU

## Estrategias: Feature Extraction vs Fine-Tuning

| | Feature Extraction | Fine-Tuning |
|---|---|---|
| **Capas congeladas** | Todas excepto la última | Solo las primeras |
| **Parámetros entrenados** | Pocos (miles) | Más (millones) |
| **Datos necesarios** | Pocos (100-1000) | Más (1000-10000+) |
| **Tiempo de entrenamiento** | Rápido | Más lento |
| **Riesgo de overfitting** | Bajo | Medio |
| **Learning rate** | Normal (1e-3) | Diferencial (1e-4 preentrenado, 1e-3 nueva capa) |
| **Cuándo usar** | Pocos datos, dominio similar | Más datos, dominio diferente |

**Regla general:**
1. Empezar con Feature Extraction (más seguro)
2. Si no es suficiente, pasar a Fine-Tuning
3. Descongelar capas progresivamente (de las últimas a las primeras)

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models

# Load pretrained ResNet50
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

print("=== ResNet50 Architecture Overview ===")
print(f"Total parameters: {sum(p.numel() for p in resnet50.parameters()):,}")

# Show top-level layer names
print("\n=== Top-level layers ===")
for name, module in resnet50.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"{name:15s} | {module.__class__.__name__:20s} | {n_params:>12,} params")

# The final FC layer
print(f"\n=== Final FC Layer ===")
print(f"Input features:  {resnet50.fc.in_features}")
print(f"Output features: {resnet50.fc.out_features} (ImageNet classes)")

# Show detailed layers in the last block (layer4)
print(f"\n=== Last residual block (layer4) ===")
for name, module in resnet50.layer4.named_modules():
    if isinstance(module, (nn.Conv2d, nn.BatchNorm2d, nn.Linear)):
        print(f"  layer4.{name}: {module}")

## Preparar dataset custom

Para simular un escenario real de transfer learning, creamos un dataset pequeño. En la práctica, usarías `ImageFolder` para cargar imágenes organizadas en carpetas por clase.

**Importante:** Siempre usar la normalización de ImageNet cuando se trabaja con modelos preentrenados en ImageNet:
- `mean = [0.485, 0.456, 0.406]`
- `std = [0.229, 0.224, 0.225]`

Y redimensionar las imágenes al tamaño que espera el modelo (224x224 para ResNet).

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

# ImageNet normalization values (MUST use these with pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transforms for pretrained models
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Use FakeData for demonstration (replace with ImageFolder for real data)
# In practice:
# train_dataset = torchvision.datasets.ImageFolder('path/to/train', transform=train_transform)
NUM_CLASSES = 5
train_full = torchvision.datasets.FakeData(
    size=500, image_size=(3, 224, 224), num_classes=NUM_CLASSES,
    transform=transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
)
test_dataset = torchvision.datasets.FakeData(
    size=100, image_size=(3, 224, 224), num_classes=NUM_CLASSES,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
)

# Split into train/val
train_size = int(0.8 * len(train_full))
val_size = len(train_full) - train_size
train_dataset, val_dataset = random_split(train_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")
print(f"Number of classes:  {NUM_CLASSES}")

# Example of how to use ImageFolder (commented out)
print("\n--- Para usar con datos reales ---")
print("""Estructura de carpetas esperada:
data/
  train/
    class_a/
      img001.jpg
      img002.jpg
    class_b/
      img001.jpg
  val/
    class_a/
      ...
    class_b/
      ...

train_dataset = torchvision.datasets.ImageFolder('data/train', transform=train_transform)
val_dataset = torchvision.datasets.ImageFolder('data/val', transform=test_transform)
""")

## Estrategia 1: Feature Extraction

En **Feature Extraction**, usamos el modelo preentrenado como un extractor de features fijo:

1. Congelar todos los pesos del modelo preentrenado (`requires_grad = False`)
2. Reemplazar la última capa FC con una nueva para nuestro número de clases
3. Solo entrenar los pesos de la nueva capa

Esto es extremadamente rápido ya que solo entrenamos unos pocos miles de parámetros.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import time
import copy

# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

def create_feature_extractor(num_classes):
    """Create a ResNet50 model for feature extraction."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    
    # Freeze ALL parameters
    for param in model.parameters():
        param.requires_grad = False
    
    # Replace the final FC layer (this one WILL be trained)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, num_classes)
    )
    
    return model

def train_model(model, train_loader, val_loader, criterion, optimizer, 
                scheduler, num_epochs, device):
    """Generic training function that returns history."""
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc = 0.0
    best_weights = None
    
    for epoch in range(num_epochs):
        start = time.time()
        
        # Training
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            train_total += labels.size(0)
            train_correct += preds.eq(labels).sum().item()
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, preds = outputs.max(1)
                val_total += labels.size(0)
                val_correct += preds.eq(labels).sum().item()
        
        # Record metrics
        train_loss /= train_total
        val_loss /= val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if scheduler:
            scheduler.step(val_loss)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
        
        elapsed = time.time() - start
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{num_epochs} ({elapsed:.1f}s) | "
                  f"Train: loss={train_loss:.4f} acc={train_acc:.4f} | "
                  f"Val: loss={val_loss:.4f} acc={val_acc:.4f}")
    
    # Load best weights
    if best_weights:
        model.load_state_dict(best_weights)
    
    return history, best_acc

# --- Feature Extraction ---
print("=== Strategy 1: Feature Extraction ===")
fe_model = create_feature_extractor(NUM_CLASSES).to(device)

# Count trainable parameters
trainable = sum(p.numel() for p in fe_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in fe_model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Only optimize the new FC layer
fe_criterion = nn.CrossEntropyLoss()
fe_optimizer = torch.optim.Adam(fe_model.fc.parameters(), lr=1e-3)
fe_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(fe_optimizer, patience=3, factor=0.5)

fe_history, fe_best_acc = train_model(
    fe_model, train_loader, val_loader,
    fe_criterion, fe_optimizer, fe_scheduler,
    num_epochs=10, device=device
)
print(f"\nBest validation accuracy (Feature Extraction): {fe_best_acc:.4f}")

## Estrategia 2: Fine-Tuning

En **Fine-Tuning**, descongelamos algunas capas del modelo preentrenado y las reentrenamos junto con la nueva capa. Usamos **differential learning rates**:

- **Learning rate bajo** para las capas preentrenadas (no queremos destruir las features aprendidas)
- **Learning rate más alto** para la nueva capa (necesita aprender desde cero)

**Típicamente descongelamos:**
- `layer4` (última capa residual) para ajustes finos
- `layer3` + `layer4` si tenemos suficientes datos
- Todo el modelo si tenemos muchos datos (> 10,000 imágenes)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def create_finetuned_model(num_classes):
    """Create a ResNet50 model for fine-tuning."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    
    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False
    
    # Unfreeze layer4 (last residual block)
    for param in model.layer4.parameters():
        param.requires_grad = True
    
    # Replace the final FC layer
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, num_classes)
    )
    
    return model

# --- Fine-Tuning ---
print("=== Strategy 2: Fine-Tuning ===")
ft_model = create_finetuned_model(NUM_CLASSES).to(device)

# Count trainable parameters
trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in ft_model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Show which layers are trainable
print("\nTrainable layers:")
for name, param in ft_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {list(param.shape)}")

# Differential learning rates
# Lower LR for pretrained layers, higher for new layers
ft_optimizer = torch.optim.Adam([
    {'params': ft_model.layer4.parameters(), 'lr': 1e-4},    # Pretrained: low LR
    {'params': ft_model.fc.parameters(), 'lr': 1e-3}         # New layer: higher LR
], weight_decay=1e-4)

ft_criterion = nn.CrossEntropyLoss()
ft_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(ft_optimizer, patience=3, factor=0.5)

ft_history, ft_best_acc = train_model(
    ft_model, train_loader, val_loader,
    ft_criterion, ft_optimizer, ft_scheduler,
    num_epochs=10, device=device
)
print(f"\nBest validation accuracy (Fine-Tuning): {ft_best_acc:.4f}")

## Comparación de resultados

Comparamos las dos estrategias visualmente para entender las diferencias en convergencia y rendimiento final.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss comparison
axes[0].plot(fe_history['train_loss'], 'b-', label='FE Train', linewidth=2)
axes[0].plot(fe_history['val_loss'], 'b--', label='FE Val', linewidth=2)
axes[0].plot(ft_history['train_loss'], 'r-', label='FT Train', linewidth=2)
axes[0].plot(ft_history['val_loss'], 'r--', label='FT Val', linewidth=2)
axes[0].set_title('Loss: Feature Extraction vs Fine-Tuning', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(fe_history['train_acc'], 'b-', label='FE Train', linewidth=2)
axes[1].plot(fe_history['val_acc'], 'b--', label='FE Val', linewidth=2)
axes[1].plot(ft_history['train_acc'], 'r-', label='FT Train', linewidth=2)
axes[1].plot(ft_history['val_acc'], 'r--', label='FT Val', linewidth=2)
axes[1].set_title('Accuracy: Feature Extraction vs Fine-Tuning', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("\n=== Resumen de Resultados ===")
print(f"{'Estrategia':<25} {'Best Val Acc':>12} {'Final Train Acc':>16} {'Final Val Loss':>15}")
print("-" * 70)
print(f"{'Feature Extraction':<25} {fe_best_acc:>12.4f} {fe_history['train_acc'][-1]:>16.4f} {fe_history['val_loss'][-1]:>15.4f}")
print(f"{'Fine-Tuning':<25} {ft_best_acc:>12.4f} {ft_history['train_acc'][-1]:>16.4f} {ft_history['val_loss'][-1]:>15.4f}")

fe_trainable = sum(p.numel() for p in create_feature_extractor(NUM_CLASSES).parameters() if p.requires_grad)
ft_trainable = sum(p.numel() for p in create_finetuned_model(NUM_CLASSES).parameters() if p.requires_grad)
print(f"\n{'Feature Extraction trainable params:':<35} {fe_trainable:>12,}")
print(f"{'Fine-Tuning trainable params:':<35} {ft_trainable:>12,}")

## Tips prácticos

### Modelos disponibles
PyTorch ofrece muchos modelos preentrenados en `torchvision.models`:

| Modelo | Params | Top-1 Acc (ImageNet) | Velocidad |
|--------|--------|---------------------|----------|
| ResNet18 | 11M | 69.8% | Rápido |
| ResNet50 | 25M | 80.9% | Medio |
| EfficientNet-B0 | 5M | 77.7% | Rápido |
| EfficientNet-B4 | 19M | 83.4% | Medio |
| ViT-B/16 | 86M | 81.1% | Lento |

### Consejos generales
1. Siempre usar la normalización de ImageNet con modelos preentrenados
2. Empezar con Feature Extraction y usar Fine-Tuning solo si es necesario
3. Usar data augmentation agresiva para prevenir overfitting
4. Guardar el mejor modelo según validation accuracy, no training accuracy

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import json

# --- Using EfficientNet (more efficient than ResNet) ---
print("=== EfficientNet-B0 ===")
efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Replace classifier
num_features = efficientnet.classifier[1].in_features
efficientnet.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, NUM_CLASSES)
)
print(f"EfficientNet-B0 parameters: {sum(p.numel() for p in efficientnet.parameters()):,}")
print(f"Classifier input features: {num_features}")

# --- Inference on a single image ---
print("\n=== Inference Example ===")

# Load a pretrained model for inference demo
inference_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
inference_model.eval()

# Standard inference transforms
inference_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create a dummy image for demonstration
import numpy as np
dummy_img = Image.fromarray(np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8))
input_tensor = inference_transform(dummy_img).unsqueeze(0)  # Add batch dim

with torch.no_grad():
    output = inference_model(input_tensor)
    probabilities = torch.softmax(output, dim=1)
    top5_prob, top5_idx = probabilities.topk(5)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"\nTop-5 predictions (on random noise image):")
for i in range(5):
    print(f"  Class {top5_idx[0][i].item():4d}: {top5_prob[0][i].item():.4f}")

# --- Save best model ---
print("\n=== Saving Model ===")

# Save the fine-tuned model with metadata
torch.save({
    'model_state_dict': ft_model.state_dict(),
    'num_classes': NUM_CLASSES,
    'architecture': 'resnet50',
    'strategy': 'fine-tuning',
    'best_val_acc': ft_best_acc,
    'normalization': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
    'input_size': 224
}, 'best_transfer_model.pt')

print("Model saved to 'best_transfer_model.pt'")
print("Saved metadata: architecture, num_classes, normalization, input_size")

# Loading the model later
print("\n--- To load the model later ---")
print("""checkpoint = torch.load('best_transfer_model.pt')
model = models.resnet50(weights=None)
model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(2048, checkpoint['num_classes']))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()""")

## Resumen: cuándo usar cada estrategia

### Árbol de decisión

```
¿Tienes datos suficientes?
├── Pocos datos (< 1000 imágenes)
│   └── Feature Extraction
│       - Congela todo el backbone
│       - Entrena solo la última capa
│       - LR: 1e-3
│       - Data augmentation agresiva
│
├── Datos moderados (1000 - 10,000)
│   └── Fine-Tuning parcial
│       - Descongela últimas 1-2 capas
│       - Differential learning rates
│       - Backbone LR: 1e-4, New layers LR: 1e-3
│
└── Muchos datos (> 10,000)
    └── Fine-Tuning completo o entrenar desde cero
        - Descongela todo
        - LR bajo: 1e-4 a 1e-5
        - Considerar entrenar desde cero si el dominio es muy diferente
```

### Checklist
1. Elegir modelo preentrenado apropiado (ResNet50 es un buen default)
2. Usar normalización de ImageNet siempre
3. Redimensionar imágenes al tamaño esperado (224x224)
4. Empezar con Feature Extraction
5. Si no es suficiente, descongelar capas progresivamente
6. Usar differential learning rates en Fine-Tuning
7. Data augmentation para combatir overfitting
8. Guardar el modelo con metadata (normalización, tamaño de entrada, clases)

### Recursos adicionales
- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [timm library](https://github.com/huggingface/pytorch-image-models): 800+ modelos preentrenados
- [Hugging Face Models](https://huggingface.co/models): modelos para visión, NLP y más